# PaliGemma 2 ile küçük deney: dil ve çözünürlük
Bu notebook iki soruyu ölçer:
1. **Dil:** Aynı soruyu Türkçe ve İngilizce sorunca doğruluk değişiyor mu? (`mix-224` modeli)
2. **Çözünürlük:** Tabela/fiş okutmada 224px ile 448px arasında fark var mı? (`mix-224` ve `mix-448`)

**Başlamadan önce**
- Runtime > Change runtime type > **T4 GPU**
- Hugging Face'te [google/paligemma2-3b-mix-224](https://huggingface.co/google/paligemma2-3b-mix-224) ve [mix-448](https://huggingface.co/google/paligemma2-3b-mix-448) sayfalarında lisansı kabul et.
- Hugging Face'ten bir **read token** oluştur, Colab'da sol menüdeki 🔑 *Secrets* bölümüne `HF_TOKEN` adıyla ekle.
- fotoğrafları `veri/` klasörüne yükle ve `etiketler.csv` dosyasını hazırla (aşağıda format var).

In [1]:
!pip -q install -U transformers accelerate pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 89.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.


In [2]:
import torch, time, re, unicodedata, difflib
import pandas as pd
from PIL import Image
from google.colab import userdata
from huggingface_hub import login
from transformers import PaliGemmaForConditionalGeneration, AutoProcessor

login(userdata.get("HF_TOKEN"))
print(torch.__version__, torch.cuda.get_device_name(0))

2.11.0+cu130 NVIDIA RTX PRO 6000 Blackwell Server Edition


## 1. Veri: `etiketler.csv`
Her satır bir örnek. Sütunlar:

| sütun | anlamı |
|---|---|
| `dosya` | `veri/` içindeki görsel adı |
| `gorev` | `vqa` (soru-cevap) veya `ocr` (yazı okuma) |
| `soru_tr` | Türkçe soru (vqa için) |
| `soru_en` | Aynı sorunun İngilizcesi (vqa için) |
| `cevap` | **Senin yazdığın** doğru cevap / görseldeki yazı |

In [76]:
%%writefile etiketler.csv
dosya,gorev,soru_tr,soru_en,cevap
tabela.jpg,ocr,,,Marmaris Datça Ula Muğla İzmir Köyceğiz Dalaman Antalya
fis.jpg,vqa,Fişin toplam tutarı nedir?,What is the total amount on the receipt?,"82,55"
nota.jpg,vqa,Bu portede kaç nota var?,How many notes are on this staff?,8
hayvan.jpg,vqa,Fotoğrafta kaç hayvan var?,How many animals are in the photo?,2
rontgen.jpg,vqa,"Görseldeki el röntgeninde kaç tane vida var? Sadece tek bir rakam yaz (Örn: 3).","How many screws are there in the hand X-ray shown in the image? Write only a single number (e.g., 3).",5
nota.jpg,vqa,"Portedeki notaları soldan sağa sırasıyla yaz (Örn: la si do).","List all notes on the staff from left to right (e.g. la si do).","A, B, C, D, E, D, C, B."

Overwriting etiketler.csv


In [77]:
# İlk çalıştırmada örnek şablon oluşturur. Kendi verinle değiştir.
import os
os.makedirs("veri", exist_ok=True)
if not os.path.exists("etiketler.csv"):
    pd.DataFrame([
        {"dosya":"tabela1.jpg","gorev":"ocr","soru_tr":"","soru_en":"","cevap":"ECZANE"},
        {"dosya":"fis1.jpg","gorev":"vqa","soru_tr":"Fişin toplam tutarı nedir?","soru_en":"What is the total amount on the receipt?","cevap":"84,50"},
    ]).to_csv("etiketler.csv", index=False)
df = pd.read_csv("etiketler.csv").fillna("")
print(len(df), "örnek"); df.head()

6 örnek


,dosya,gorev,soru_tr,soru_en,cevap
0,tabela.jpg,ocr,,,Marmaris Datça Ula Muğla İzmir Köyceğiz Dalama...
1,fis.jpg,vqa,Fişin toplam tutarı nedir?,What is the total amount on the receipt?,"82,55"
2,nota.jpg,vqa,Bu portede kaç nota var?,How many notes are on this staff?,8
3,hayvan.jpg,vqa,Fotoğrafta kaç hayvan var?,How many animals are in the photo?,2
4,rontgen.jpg,vqa,Görseldeki el röntgeninde kaç tane vida var? S...,How many screws are there in the hand X-ray sh...,5


## 2. Yardımcı fonksiyonlar

In [50]:
def normalize(s):
    # Türkçe I/İ sorununu önlemek için önce özel harfleri çevir, sonra küçült
    s = s.replace("İ","i").replace("I","ı")
    s = unicodedata.normalize("NFKC", s).casefold()
    s = re.sub(r"[^\w\s,.]", "", s)
    return re.sub(r"\s+", " ", s).strip()

def levenshtein(a, b):
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j]+1, cur[j-1]+1, prev[j-1]+(ca!=cb)))
        prev = cur
    return prev[-1]

def cer(pred, ref):
    pred, ref = normalize(pred), normalize(ref)
    return levenshtein(pred, ref) / max(len(ref), 1)

def contains_match(pred, ref):
    p = normalize(pred).replace(",", ".")
    r = normalize(ref).replace(",", ".")
    return r in p

In [51]:
def load(model_id):
    model = PaliGemmaForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=torch.bfloat16, device_map="cuda").eval()
    processor = AutoProcessor.from_pretrained(model_id)
    return model, processor

@torch.inference_mode()
def ask(model, processor, image, prompt, max_new_tokens=60):
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(torch.bfloat16).to(model.device)
    n_in = inputs["input_ids"].shape[-1]          # resim token'ları + metin
    t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    dt = time.time() - t0
    text = processor.decode(out[0][n_in:], skip_special_tokens=True)
    return text.strip(), n_in, dt

## Deney 1: Türkçe soru mu, İngilizce soru mu? (224px)
PaliGemma "mix" modelleri istem başında görev ve dil kodu bekler: `answer en ...` / `answer tr ...`.
Soru metni Türkçe, kod da `tr` olursa fark ediyor mu?

In [61]:
model, proc = load("google/paligemma2-3b-mix-224")
vqa = df[df.gorev=="vqa"]
rows = []
for _, r in vqa.iterrows():
    img = Image.open(f"veri/{r.dosya}").convert("RGB")
    for kosul, prompt in [("en", f"answer en {r.soru_en}"),
                          ("tr", f"answer tr {r.soru_tr}")]:
        pred, n_in, dt = ask(model, proc, img, prompt)
        rows.append(dict(dosya=r.dosya, kosul=kosul, tahmin=pred, cevap=r.cevap,
                         dogru=contains_match(pred, r.cevap), token=n_in, sn=round(dt,2)))
res1 = pd.DataFrame(rows)
res1

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

[transformers] You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
[transformers] You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
[transformers] You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many i

,dosya,kosul,tahmin,cevap,dogru,token,sn
0,fis.jpg,en,82.55,"82,55",True,269,0.08
1,fis.jpg,tr,"82,55","82,55",True,269,0.08
2,nota.jpg,en,5,8,False,268,0.03
3,nota.jpg,tr,4,8,False,267,0.03
4,hayvan.jpg,en,2,2,True,268,0.03
5,hayvan.jpg,tr,2,2,True,267,0.03
6,rontgen.jpg,en,5,5,True,289,0.03
7,rontgen.jpg,tr,3,5,False,286,0.03
8,nota.jpg,en,"D, E, F, G, A, B.","A, B, C, D, E, D, C, B.",False,279,0.15
9,nota.jpg,tr,"D, E, F, G, A, B.","A, B, C, D, E, D, C, B.",False,280,0.15


In [62]:
print(res1.groupby("kosul").dogru.agg(["mean","sum","count"]).rename(
    columns={"mean":"doğruluk","sum":"doğru","count":"toplam"}))

       doğruluk  doğru  toplam
kosul                         
en          0.6      3       5
tr          0.4      2       5


## Deney 2: 224px mi, 448px mi? (OCR)
Aynı görseller, aynı istem (`ocr`), iki farklı çözünürlük. Metrik: karakter hata oranı (CER, düşük iyi). İstem başındaki token sayısını da kaydediyoruz (224px için 256 resim token'ı, 448px için 1024 olmalı).

In [78]:
ocr = df[df.gorev=="ocr"]
rows = []
for model_id, res_adi in [("google/paligemma2-3b-mix-224","224"),
                          ("google/paligemma2-3b-mix-448","448")]:
    del model; torch.cuda.empty_cache()
    model, proc = load(model_id)
    for _, r in ocr.iterrows():
        img = Image.open(f"veri/{r.dosya}").convert("RGB")
        pred, n_in, dt = ask(model, proc, img, "ocr")
        rows.append(dict(dosya=r.dosya, cozunurluk=res_adi, tahmin=pred, cevap=r.cevap,
                         cer=round(cer(pred, r.cevap),3), tam_dogru=normalize(pred)==normalize(r.cevap),
                         token=n_in, sn=round(dt,2)))
res2 = pd.DataFrame(rows)
res2

Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

[transformers] You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.


Loading weights:   0%|          | 0/727 [00:00<?, ?it/s]

[transformers] You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.


,dosya,cozunurluk,tahmin,cevap,cer,tam_dogru,token,sn
0,tabela.jpg,224,Marmaris\nUlu Muğla İzmir\nKayseri\nKayseri\nA...,Marmaris Datça Ula Muğla İzmir Köyceğiz Dalama...,0.309,False,259,0.24
1,tabela.jpg,448,Marmaris Datça\nUla Muğla İzmir\nKöyceğiz Dala...,Marmaris Datça Ula Muğla İzmir Köyceğiz Dalama...,0.000,True,1027,0.26


In [79]:
print(res2.groupby("cozunurluk").agg(ort_CER=("cer","mean"), tam_dogru=("tam_dogru","sum"),
      ort_token=("token","mean"), ort_sn=("sn","mean"), n=("cer","count")).round(3))

            ort_CER  tam_dogru  ort_token  ort_sn  n
cozunurluk                                          
224           0.309          0      259.0    0.24  1
448           0.000          1     1027.0    0.26  1


## Sonuçları kaydet

In [80]:
res1.to_csv("sonuclar_dil.csv", index=False)
res2.to_csv("sonuclar_cozunurluk.csv", index=False)
res1[~res1.dogru]

,dosya,kosul,tahmin,cevap,dogru,token,sn
2,nota.jpg,en,5,8,False,268,0.03
3,nota.jpg,tr,4,8,False,267,0.03
7,rontgen.jpg,tr,3,5,False,286,0.03
8,nota.jpg,en,"D, E, F, G, A, B.","A, B, C, D, E, D, C, B.",False,279,0.15
9,nota.jpg,tr,"D, E, F, G, A, B.","A, B, C, D, E, D, C, B.",False,280,0.15
